# (6) Uncertainty Quantification

The two surrogate networks from Chapter 4 (`EnvelopeMLP` and `MapDecoder`) are deterministic
regressors — they emit a single point estimate for every design vector, with no notion of
how confident the model is in that estimate.

For a thesis result this is uncomfortable. The training set covers 2511 LHS-sampled designs
inside a fixed 6-D box; if a downstream user queries the surrogate at a design that lies
near the edge of (or outside) that box, the network will still return *some* prediction —
but the prediction may be far from the true FEMM answer. Without uncertainty estimates,
there is no automatic way to flag those cases.

This chapter wires uncertainty estimation onto the **existing trained weights** — no
retraining — by treating the 5 cross-validation fold models as a small **deep ensemble**.
The disagreement between them, at each design and each $(N, T)$ cell of the predicted
$\eta$ map, is a calibrated proxy for epistemic uncertainty.


## 6.1  Aleatoric vs epistemic uncertainty

Two flavours of uncertainty matter for surrogate models:

- **Aleatoric** (data noise) — irreducible randomness in the labels. For our dataset the
  ground-truth $\eta$ values come from a *deterministic* FEMM solve, so aleatoric noise
  is effectively zero at the resolution we care about. We will not model it explicitly.
- **Epistemic** (model knowledge) — uncertainty arising because the model has only seen
  2511 designs out of an infinite design space. This shrinks as more data is added and is
  what we *can* measure usefully here.

A common, cheap proxy for epistemic uncertainty is **deep-ensemble disagreement**: train
$M$ models from different random initialisations on the same data, predict with all of
them at inference, and use the spread as the uncertainty signal. We already trained 5
models — one per CV fold — so we can use them as a 5-member ensemble for free.

**Honesty caveat.** A true deep ensemble trains each member on *all* the data with
different random initialisations. Our 5 fold models were each trained on overlapping
80% subsets, so the disagreement between them is partly initialisation variance and
partly data-resampling variance. The spread is therefore biased *downward* compared to
a clean deep ensemble. We will still get a useful calibrated signal, just one that
tends to under-estimate true epistemic uncertainty. We return to this in §6.8.


## 6.2  Loading the fold models

Each `weights_fold{k}.pt` is a `state_dict` matching a specific `MapDecoder`
architecture from Chapter 4. We instantiate the matching topology, load each set
of weights, and keep them in a list for batched prediction.

The cell below loads **two** ensembles in parallel:

- **width-4× (primary, production)** — 633 k parameters, OOF $R^2 = 0.973$,
  the recommended decoder per §4.7.3 / §7.7.
- **baseline (reference)** — 157 k parameters, OOF $R^2 = 0.913$, the original
  Phase-5B topology.

Subsequent figures and statistics use the width-4× ensemble; the baseline copy
is kept around so the §6.5.1 comparison can show what tightened by switching to
the better-fit decoder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

from data_gen.ml.models import MapDecoder, EnvelopeMLP, MAPDECODER_PRESETS

ROOT = Path('data_gen/outputs')
MAP_DIR_BASELINE = ROOT / 'phase5_map_train'                    # 157 k params
MAP_DIR_W4X      = ROOT / 'phase5_map_train_arch' / 'width4x'   # 633 k params
ENV_DIR          = ROOT / 'phase5_envelope_train'

# Both runs share train/val splits + input standardisation (verified). Use the
# width-4x predictions.npz as the source of truth — it is now the production
# decoder per §4.7.3 / §7.7.
map_pred = np.load(MAP_DIR_W4X / 'predictions.npz', allow_pickle=False)
env_pred = np.load(ENV_DIR / 'predictions.npz', allow_pickle=False)
master   = np.load(ROOT / 'dataset_train' / 'master_dataset.npz', allow_pickle=False)

x_mean, x_std = map_pred['input_mean'], map_pred['input_std']
X = master['design_params'].astype(np.float32)
X_std = ((X - x_mean) / x_std).astype(np.float32)
fold_id = map_pred['fold_id']           # which fold validated each design (1..5)

# Load the 5 fold models for each architecture. We carry both so the §6.5
# comparison can show the UQ tightening that §7.8 promised.
map_models = []                              # primary: width-4x
for k in range(1, 6):
    m = MapDecoder(input_dim=6, **MAPDECODER_PRESETS['width4x'])
    sd = torch.load(MAP_DIR_W4X / f'weights_fold{k}.pt', map_location='cpu',
                    weights_only=True)
    m.load_state_dict(sd, strict=True)
    m.eval()
    map_models.append(m)

map_models_baseline = []                     # reference: 157 k baseline
for k in range(1, 6):
    m = MapDecoder(input_dim=6)
    sd = torch.load(MAP_DIR_BASELINE / f'weights_fold{k}.pt', map_location='cpu',
                    weights_only=True)
    m.load_state_dict(sd, strict=True)
    m.eval()
    map_models_baseline.append(m)

print(f'loaded {len(map_models)} width-4x   MapDecoder fold models '
      f'({map_models[0].n_params} params each)')
print(f'loaded {len(map_models_baseline)} baseline MapDecoder fold models '
      f'({map_models_baseline[0].n_params} params each)')

## 6.3  Forward pass through the ensemble

Run all 2511 standardised design vectors through every fold model. Store the predictions
as a `(5, 2511, 60, 80)` tensor — five 60×80 maps per design.

**Honest OOF ensemble.** For each design $i$ we exclude the one fold model that *trained*
on it (the model that held it out has it in its validation set, never training, so it is
kept). This produces a 4-member ensemble per design — a fair test of epistemic uncertainty.
The fifth (held-out) model's prediction is identical to `oof_pred` from Chapter 7 and is
kept as the point estimate.


In [ ]:
X_t = torch.as_tensor(X_std, dtype=torch.float32)
with torch.no_grad():
    ensemble_preds = torch.stack([m(X_t) for m in map_models], dim=0).numpy()
    ensemble_preds_baseline = torch.stack(
        [m(X_t) for m in map_models_baseline], dim=0).numpy()

n_designs = X.shape[0]

def honest_stats(preds_array):
    """Compute honest CV-fold ensemble: for each design i, mean+std over the
    4 folds that did NOT train on i."""
    mean = np.zeros((n_designs, 60, 80), dtype=np.float32)
    std  = np.zeros((n_designs, 60, 80), dtype=np.float32)
    for i in range(n_designs):
        held_out_fold = fold_id[i]
        keep = [k for k in range(5) if (k + 1) != held_out_fold]
        members = preds_array[keep, i]                       # (4, 60, 80)
        mean[i] = members.mean(axis=0)
        std[i]  = members.std(axis=0, ddof=1)
    return mean, std

honest_mean, honest_std = honest_stats(ensemble_preds)
honest_mean_baseline, honest_std_baseline = honest_stats(ensemble_preds_baseline)

mask = map_pred['mask']
print(f'ensemble shapes (width-4x): preds={ensemble_preds.shape}, '
      f'mean={honest_mean.shape}, std={honest_std.shape}')
print(f'median per-cell std  (width-4x): {np.median(honest_std[mask]):.4f}')
print(f'median per-cell std  (baseline): {np.median(honest_std_baseline[mask]):.4f}')
print(f'max per-cell std     (width-4x): {honest_std[mask].max():.4f}')

## 6.4  Per-design uncertainty maps

Pick four designs across the $T_\text{max}$ spread and show, side by side, the ensemble
mean prediction, the per-cell std (uncertainty), and the actual absolute error against the
FEMM ground truth. A well-behaved UQ method has the std map qualitatively tracking the
error map — light where the model is confident *and* correct, dark where it is uncertain
*and* wrong.


In [ ]:
y_true = map_pred['oof_true']
mask   = map_pred['mask']
T_max_per = master['T_max_Nm']
N_axis = master['N_norm_grid']
T_axis = master['T_norm_grid']

order = np.argsort(T_max_per)
picks = [order[0], order[len(order)//3], order[2*len(order)//3], order[-1]]

fig, axes = plt.subplots(len(picks), 3, figsize=(14, 4*len(picks)))
for row, idx in enumerate(picks):
    m = mask[idx]
    mean_ = np.where(m, honest_mean[idx], np.nan)
    std_  = np.where(m, honest_std[idx],  np.nan)
    err   = np.where(m, np.abs(honest_mean[idx] - y_true[idx]), np.nan)
    Tm = T_max_per[idx]

    cells_ = [
        (mean_, f'ensemble mean ($T_\\mathrm{{max}}$={Tm:.1f} Nm)', 'viridis', 0, 1.0),
        (std_,  'per-cell std (uncertainty)',                        'magma',   0, 0.15),
        (err,   'abs error vs FEMM truth',                           'magma',   0, 0.3),
    ]
    for col, (img, title, cmap, vmin, vmax) in enumerate(cells_):
        ax = axes[row, col]
        im = ax.imshow(img, origin='lower', aspect='auto',
                       extent=[N_axis[0], N_axis[-1], T_axis[0], T_axis[-1]],
                       cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_xlabel('$N / N_\\mathrm{max}$')
        ax.set_ylabel('$T / T_\\mathrm{max}$')
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


Visual inspection: the std map is typically highest near the operating-envelope boundary
(cells that are inside the envelope for some designs and outside for others) and along
the low-speed/high-torque corner where Cu loss dominates. The actual error map shows the
same hot spots — a sign that the ensemble disagreement is tracking a real signal, not
noise.


## 6.5  Calibration analysis

A useful uncertainty estimator has to be **calibrated** — when the model says it is
uncertain, it should actually be more often wrong. We check four properties:

1. **Scatter** of predicted std vs actual abs error, across all (cell, design) pairs.
2. **Binned calibration curve**: mean abs error per std bucket.
3. **Reliability diagram**: under a Gaussian assumption, what fraction of true values
   lie within $\pm 1\sigma$ and $\pm 2\sigma$ of the ensemble mean? Should be 68% and
   95% for a well-calibrated UQ.
4. **Sharpness**: median predicted std, and the per-design distribution of mean std.
   A useful UQ is both calibrated *and* sharp — wide intervals everywhere would be
   calibrated but useless.


In [ ]:
# Flatten all (cell, design) pairs that are inside an envelope.
valid = mask
pred_std_flat = honest_std[valid]
abs_err_flat  = np.abs(honest_mean - y_true)[valid]
residual_flat = (y_true - honest_mean)[valid]

# Subsample for the scatter (5000 random points).
rng = np.random.default_rng(0)
n_show = min(5000, pred_std_flat.size)
sel = rng.choice(pred_std_flat.size, n_show, replace=False)

# Binned calibration curve.
n_bins = 10
bin_edges = np.quantile(pred_std_flat, np.linspace(0, 1, n_bins + 1))
bin_idx = np.clip(np.digitize(pred_std_flat, bin_edges[1:-1]), 0, n_bins - 1)
bin_pred_std = np.array([pred_std_flat[bin_idx == b].mean() for b in range(n_bins)])
bin_actual_mae = np.array([abs_err_flat[bin_idx == b].mean() for b in range(n_bins)])

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

ax = axes[0, 0]
ax.scatter(pred_std_flat[sel], abs_err_flat[sel], s=3, alpha=0.15)
lim = max(pred_std_flat.max(), abs_err_flat.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='$|err| = \\sigma$')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('Predicted std $\\sigma$')
ax.set_ylabel('Actual $|\\mathrm{error}|$')
ax.set_title(f'Calibration scatter  ({n_show} random points)')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(bin_pred_std, bin_actual_mae, 'o-', lw=2)
max_xy = max(bin_pred_std.max(), bin_actual_mae.max()) * 1.05
ax.plot([0, max_xy], [0, max_xy], 'k--', lw=1, label='ideal')
ax.set_xlim(0, max_xy); ax.set_ylim(0, max_xy)
ax.set_xlabel('Mean predicted std (per bin)')
ax.set_ylabel('Mean actual |error| (per bin)')
ax.set_title(f'Binned calibration curve ({n_bins} std-quantile bins)')
ax.legend(); ax.grid(alpha=0.3)

# Reliability diagram: empirical coverage at multiples of sigma.
ks = np.linspace(0.25, 3.0, 12)
from math import erf, sqrt
nominal = np.array([erf(k / sqrt(2)) for k in ks])
empirical = np.array([
    (np.abs(residual_flat) <= k * pred_std_flat).mean() for k in ks
])
ax = axes[1, 0]
ax.plot(nominal, empirical, 'o-', lw=2, label='ensemble')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='ideal')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel('Nominal coverage (Gaussian)')
ax.set_ylabel('Empirical coverage')
ax.set_title('Reliability diagram')
ax.legend(); ax.grid(alpha=0.3)

# Sharpness.
per_design_mean_std = np.array([
    honest_std[i][mask[i]].mean() if mask[i].any() else np.nan
    for i in range(n_designs)
])
ax = axes[1, 1]
ax.hist(per_design_mean_std, bins=40, edgecolor='black', alpha=0.7)
med = float(np.nanmedian(per_design_mean_std))
ax.axvline(med, color='r', lw=1.5, label=f'median = {med:.4f}')
ax.set_xlabel('Per-design mean predicted std')
ax.set_ylabel('Designs')
ax.set_title('Sharpness: per-design uncertainty distribution')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

from scipy.stats import spearmanr
rho, _ = spearmanr(pred_std_flat[sel], abs_err_flat[sel])
print(f'Spearman rho (pred std, |err|): {rho:+.3f}')
print(f'Median per-cell std:            {float(np.median(pred_std_flat)):.4f}')
print(f'Empirical 1-sigma coverage:     {(np.abs(residual_flat) <= pred_std_flat).mean()*100:.1f}%'
      f'  (nominal 68.3%)')
print(f'Empirical 2-sigma coverage:     {(np.abs(residual_flat) <= 2*pred_std_flat).mean()*100:.1f}%'
      f'  (nominal 95.4%)')


How to read these:

- **Scatter (top-left).** Positive correlation between predicted std and actual error
  confirms the ensemble disagreement contains real signal. The cloud is below the
  $y = \sigma$ line because the ensemble *under-estimates* uncertainty — a known
  consequence of CV-fold ensembling (see §6.8).
- **Binned calibration (top-right).** Points should fall on or near the $y = x$ line if
  predicted std and actual MAE move together. A flat line at large std means the model
  cries wolf in low-error regions; a steep curve means it is over-confident in high-error
  regions.
- **Reliability diagram (bottom-left).** A curve that falls *below* the ideal indicates
  under-estimation of uncertainty (empirical coverage at nominal 68% is less than 68%).
  This is what we expect from CV-fold ensembling.
- **Sharpness (bottom-right).** Most designs sit at very small mean std (the model is
  confident everywhere); the tail at high mean std identifies the designs the ensemble
  most disagrees on — natural candidates for FEMM follow-up (§6.7).


### 6.5.1  Baseline vs width-4× — what tightened?

The §7.8 promise was that the CV-fold ensemble UQ would *tighten* on the better-fit
decoder. Running the same honest-ensemble pipeline on both architectures gives
the side-by-side calibration metrics below.

What we expect to see, given §4.7.3's $R^2 = 0.913 \to 0.973$ shift:

- **Mean $|\text{error}|$** should fall substantially — the better decoder makes
  individually better predictions on every held-out fold.
- **Per-cell $\sigma$** may not shrink: the width-4× folds *agree more* with each
  other (lower ensemble variance) only if the 4 retained folds find similar
  optima. If they diverge more (richer loss surface), $\sigma$ could even grow.
- **Spearman $\rho(\sigma, |\text{error}|)$** is the cleanest UQ-quality signal:
  it asks whether the cells the ensemble flags as uncertain are *actually* the
  cells where the prediction is wrong. The baseline ensemble achieved
  $\rho \approx +0.55$ (per §6.5).
- **Coverage** should move toward nominal (1-σ ≈ 68 %, 2-σ ≈ 95 %). The baseline
  honest ensemble was severely under-confident (only ~12 % of cells inside 1 σ),
  so any tightening of $|\text{error}|$ without a matching shrink in $\sigma$
  pushes coverage in the right direction.

The cell below computes both columns and prints the difference.

In [ ]:
# Compare honest-ensemble UQ on the baseline (157 k) and width-4x (633 k) models.
from scipy.stats import spearmanr
from math import erf, sqrt

def uq_summary(hmean, hstd, y_true_, mask_):
    valid = mask_
    pred_std = hstd[valid]
    abs_err  = np.abs(hmean - y_true_)[valid]
    residual = (y_true_ - hmean)[valid]
    rho_full, _ = spearmanr(pred_std, abs_err)
    return {
        'median_sigma':   float(np.median(pred_std)),
        'mean_sigma':     float(np.mean(pred_std)),
        'mean_abs_err':   float(np.mean(abs_err)),
        'rho_sigma_err':  float(rho_full),
        'cov_1sigma_pct': float((np.abs(residual) <= pred_std).mean() * 100),
        'cov_2sigma_pct': float((np.abs(residual) <= 2 * pred_std).mean() * 100),
    }

# y_true and mask are defined in cell 7 (§6.4). Recompute here in case execution
# order differs.
y_true_ = map_pred['oof_true']
mask_ = map_pred['mask']

summary_w4x      = uq_summary(honest_mean,          honest_std,          y_true_, mask_)
summary_baseline = uq_summary(honest_mean_baseline, honest_std_baseline, y_true_, mask_)

# nominal 1-sigma = 68.3%, 2-sigma = 95.4%
print('Honest CV-fold ensemble UQ — baseline (157 k) vs width-4x (633 k):\n')
fmt = '{:<22s} {:>14s} {:>14s} {:>10s}'
print(fmt.format('metric', 'baseline', 'width-4x', 'Δ'))
print('-' * 64)
for key, label, fmt_v in [
    ('median_sigma',   'Median predicted σ',     '{:.4f}'),
    ('mean_sigma',     'Mean predicted σ',       '{:.4f}'),
    ('mean_abs_err',   'Mean |error|',           '{:.4f}'),
    ('rho_sigma_err',  'Spearman ρ(σ, |err|)',  '{:+.3f}'),
    ('cov_1sigma_pct', '1-σ coverage (nom 68%)', '{:.1f}%'),
    ('cov_2sigma_pct', '2-σ coverage (nom 95%)', '{:.1f}%'),
]:
    b = summary_baseline[key]
    w = summary_w4x[key]
    delta = w - b
    print(fmt.format(label, fmt_v.format(b), fmt_v.format(w), fmt_v.format(delta).lstrip('+')))
print()

# Per-cell error reduction (does the width-4x ensemble produce smaller errors on
# the SAME held-out cells where baseline struggled?).
err_base = np.abs(honest_mean_baseline - y_true_)[mask_]
err_w4x  = np.abs(honest_mean          - y_true_)[mask_]
print(f'Per-cell |error| reduction: '
      f'mean |err| {err_base.mean():.4f} (baseline) → {err_w4x.mean():.4f} (width-4x) '
      f'({(1 - err_w4x.mean() / err_base.mean()) * 100:.1f}% reduction)')

## 6.6  Envelope-net uncertainty

Repeat the procedure on the `EnvelopeMLP` for $T_\text{max}$. Five fold models, one
scalar prediction per design per model.


In [ ]:
env_models = []
for k in range(1, 6):
    m = EnvelopeMLP(input_dim=6, hidden=(32, 32), output_dim=2)
    sd = torch.load(ENV_DIR / f'weights_fold{k}.pt', map_location='cpu',
                    weights_only=True)
    m.load_state_dict(sd, strict=True)
    m.eval()
    env_models.append(m)

with torch.no_grad():
    env_ens = torch.stack([m(X_t) for m in env_models], dim=0).numpy()  # (5, 2511, 2)

# Envelope targets were standardised; un-standardise back to physical units.
y_mean = env_pred['target_mean']
y_std  = env_pred['target_std']
env_ens_phys = env_ens * y_std + y_mean

env_fold_id = env_pred['fold_id']
env_honest_mean = np.zeros((n_designs, 2), dtype=np.float32)
env_honest_std  = np.zeros((n_designs, 2), dtype=np.float32)
for i in range(n_designs):
    keep = [k for k in range(5) if (k + 1) != env_fold_id[i]]
    env_honest_mean[i] = env_ens_phys[keep, i].mean(axis=0)
    env_honest_std[i]  = env_ens_phys[keep, i].std(axis=0, ddof=1)

t_true = env_pred['oof_true'][:, 1]
t_mean = env_honest_mean[:, 1]
t_std  = env_honest_std[:, 1]
t_err  = np.abs(t_mean - t_true)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
ax.errorbar(t_true, t_mean, yerr=t_std, fmt='o', ms=3, alpha=0.4, ecolor='gray',
            elinewidth=0.5)
lim = [0, max(t_true.max(), t_mean.max()) * 1.05]
ax.plot(lim, lim, 'k--', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('True $T_\\mathrm{max}$ (Nm)')
ax.set_ylabel('Predicted $T_\\mathrm{max}$ (Nm) $\\pm \\sigma$')
ax.set_title('Envelope net: prediction with ensemble error bars')
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(t_std, t_err, s=10, alpha=0.5)
max_xy = max(t_std.max(), t_err.max()) * 1.05
ax.plot([0, max_xy], [0, max_xy], 'k--', lw=1, label='$|err| = \\sigma$')
ax.set_xlim(0, max_xy); ax.set_ylim(0, max_xy)
ax.set_xlabel('Predicted std $\\sigma$ (Nm)')
ax.set_ylabel('Actual $|\\mathrm{error}|$ (Nm)')
ax.set_title('Envelope net: $\\sigma$ vs $|\\mathrm{error}|$')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

rho_env, _ = spearmanr(t_std, t_err)
print(f'Envelope Spearman rho (sigma, |err|): {rho_env:+.3f}')
print(f'Median sigma: {np.median(t_std):.3f} Nm  vs  median |err|: {np.median(t_err):.3f} Nm')


## 6.7  Decision rule — which designs to FEMM-validate first

Practical use: if a downstream user wants surrogate predictions on $K$ new designs and
has budget for $k \ll K$ FEMM follow-up runs, the highest-value choice is to FEMM the
designs where the ensemble disagrees most. Below we rank the 2511 training designs by
mean predicted std across the $\eta$ map and show the top-5 — the designs where the
ensemble is least sure of itself.


In [ ]:
per_design_mean_std_clean = per_design_mean_std.copy()
per_design_mean_std_clean[np.isnan(per_design_mean_std_clean)] = -1
top5 = np.argsort(per_design_mean_std_clean)[-5:][::-1]

rows = []
for i in top5:
    rows.append({
        'design_idx': int(i),
        'T_max_Nm': float(T_max_per[i]),
        'mean_std': float(per_design_mean_std[i]),
        'mean_abs_err': float(np.mean(np.abs(honest_mean[i] - y_true[i])[mask[i]])),
    })
print('Top-5 most-uncertain designs in the training set:')
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(1, len(top5), figsize=(4*len(top5), 4))
for ax, idx in zip(axes, top5):
    img = np.where(mask[idx], honest_std[idx], np.nan)
    im = ax.imshow(img, origin='lower', aspect='auto',
                   extent=[N_axis[0], N_axis[-1], T_axis[0], T_axis[-1]],
                   cmap='magma', vmin=0, vmax=float(np.nanmax(img)))
    ax.set_xlabel('$N / N_\\mathrm{max}$')
    ax.set_ylabel('$T / T_\\mathrm{max}$')
    ax.set_title(f'design {int(idx)}\n$T_\\mathrm{{max}}$={T_max_per[idx]:.1f} Nm')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle('Top-5 most-uncertain designs — std maps')
plt.tight_layout()
plt.show()


:::{admonition} Future work — three independent UQ signals beyond the CV-fold ensemble
:class: tip
**Expected wall:** ~2–3 hours total on CPU.

- **Independent 5-model ensemble** (~1.5–2 hours): retrain 5 width-4× `MapDecoder`s with different random init *on the same full 2511-design dataset* (no fold structure). Compute per-cell σ over the 5 independent predictions.
- **MC-Dropout pass** (~10 minutes): add `nn.Dropout` after each ReLU in the head/decoder, retrain ONE model, sample 50 forward passes per design with dropout left active.
- **Quantile-regression head** (~30 minutes): replace the final ConvTranspose with two parallel heads predicting the 0.1 and 0.9 quantiles; compute per-cell σ from the half-interquantile range.

**Expected advantage:** three concurring estimates would let us cite a *defended* coverage % rather than the single CV-fold-ensemble number. The §6.5.1 comparison already showed the honest CV-fold ensemble under-estimates 1-σ coverage substantially (~19 % vs nominal 68 % on width-4×); whether that under-estimation is intrinsic to the CV-fold construction (likely) or a width-4× artefact (less likely) would be resolved by the three-signal cross-check. The independent ensemble in particular gives true epistemic UQ unconstrained by fold structure.
:::

## 6.8  Limitations and next steps

**What the CV-fold ensemble is and isn't.**

- *Is*: a free, real-data UQ signal — Spearman correlation between predicted std and
  actual abs error is positive, the reliability diagram is monotone, the sharpness is
  bounded. Good enough to identify high-risk predictions and guide where to spend more
  FEMM cycles.
- *Isn't*: well-calibrated in the strict sense. Because the 5 fold models share 60% of
  their training data pairwise (each fold drops a different 20%), their disagreement
  under-estimates the true epistemic uncertainty. Empirical 1-σ coverage will sit below
  the nominal 68%, which is visible in the reliability diagram from §6.5.

**Sensible upgrades, all cheap to add.**

1. **Independent random-init ensemble.** Re-run `run_phase5_map.py` with five different
   seeds, each trained on the *full* 2511 designs. Replaces the fold ensemble; properly
   captures init variance. Cost: 5 × ~30 min ≈ 2.5 h on CPU at the current dataset size.
2. **MC Dropout.** Add `nn.Dropout(p=0.1)` after each ReLU in `MapDecoder`, retrain,
   then at inference draw $M$ stochastic forward passes per input. Same calibration
   analysis pipeline applies unchanged. Cost: one retrain + a forward-pass-time multiplier.
3. **Bayesian last layer.** Keep all but the final layer deterministic and place a Gaussian
   prior on the last layer's weights; predictive distribution is closed-form Gaussian.
   The cheapest principled Bayesian option.
4. **Quantile-regression head.** Replace the sigmoid scalar output with three heads
   (10th, 50th, 90th quantile) trained with pinball loss. Gives prediction intervals
   directly without ensembling.

Each of these is a small additional `data_gen/ml/` script and a single notebook section;
the calibration analysis from §6.5 transplants unchanged. For the current thesis chapter
the CV-fold ensemble is the right level of detail — real signal, no retraining, clear
honesty about the bias.
